In [ ]:
import numpy as np

# Constants
amu_MeV = 931.49410242  # 1 amu in MeV/c^2
hbarc = 197.3269804  # MeV·fm
alpha = 1/137.035999084  # fine structure constant
e_sq = 1.4399764  # e^2 in MeV·fm
m_p = 938.27208816  # proton mass in MeV/c^2
m_n = 939.56542052  # neutron mass in MeV/c^2

In [ ]:
# Atomic masses in amu (from AME2020)
masses = {
    '1H': 1.00782503223,
    'n': 1.00866491595,
    '45Sc': 44.95590828,
    '45Ti': 44.95812562,
    '48Ti': 47.94794198,
    '48V': 47.9522535,
    '51V': 50.94395704,
    '51Cr': 50.94820869,
    '52Cr': 51.94050623,
    '52Mn': 51.9455639,
    '55Mn': 54.93804391,
    '55Fe': 54.94229166,
    '56Fe': 55.93493633,
    '56Co': 55.9398388,
    '59Co': 58.93319429,
    '59Ni': 58.9343462,
    '62Ni': 61.92834488,
    '62Cu': 61.9340583,
    '63Cu': 62.92959772,
    '63Zn': 62.9332116,
    '66Zn': 65.92603381,
    '66Ga': 65.9318084}

In [ ]:

def calculate_q_value(target, product):
    """Calculate Q-value in MeV (corrected sign conve ntion)"""
    M_target = masses[target] * amu_MeV
    M_product = masses[product] * amu_MeV
    M_p = m_p
    M_n = m_n

    Q = (M_target + M_p - M_product - M_n)  # Corrected for (p,n) reactions
    return Q

def calculate_thresholds(Q):
    """Calculate lab and CM threshold energies (always positive)"""
    E_cm = -Q if Q < 0 else 0  # Only exists if Q < 0
    E_lab = -Q * (M_target + m_p) / M_target if Q < 0 else 0
    return E_lab, E_cm

def sommerfeld_parameter(Z1, Z2, E_cm):
    if E_cm <= 0:
        return np.nan
    mu = reduced_mass(m_p, M_target)[0]
    k = np.sqrt(2 * mu * E_cm) / hbarc
    return Z1 * Z2 * e_sq*mu / (hbarc*hbarc*k)  # ✅ removed extra hbarc


def reduced_mass(m1, m2):
    mu = (m1 * m2) / (m1 + m2)
    return mu, mu / amu_MeV  # Return in MeV and amu

def coulomb_barrier(Z1, Z2, A1, A2):
    R = 1.2 * (A1**(1/3) + A2**(1/3))  # nuclear radii approx. in fm
    V_c = Z1 * Z2 * e_sq / R  # Coulomb potential
    return V_c

def gamow_energy(Z1, Z2, mu):
    return (2 * np.pi * Z1 * Z2 * e_sq)**2 * mu / (2 * hbarc**2)


In [ ]:
reactions = [
    ('45Sc', '45Ti'), ('48Ti', '48V'), ('51V', '51Cr'),
    ('52Cr', '52Mn'), ('55Mn', '55Fe'), ('56Fe', '56Co'),
    ('59Co', '59Ni'), ('62Ni', '62Cu'), ('63Cu', '63Zn'),
    ('66Zn', '66Ga')
]

results = []

for target, product in reactions:
    # Get atomic numbers and mass numbers
    A1 = int(target[:2]) if target[1].isdigit() else int(target[0])
    Z1 = {
        'Sc': 21, 'Ti': 22, 'V': 23, 'Cr': 24, 'Mn': 25,
        'Fe': 26, 'Co': 27, 'Ni': 28, 'Cu': 29, 'Zn': 30
    }[target[2:] if target[1].isdigit() else target[1:]]

    Z2 = Z1 + 1  # Since (p,n) increases Z by 1

    # Calculations
    Q = calculate_q_value(target, product)
    M_target = masses[target] * amu_MeV
    E_lab, E_cm = calculate_thresholds(Q)
    V_c = coulomb_barrier(1, Z1, 1, A1)
    mu_MeV, mu_amu = reduced_mass(m_p, M_target)
    ratio = E_cm / E_lab
    eta = sommerfeld_parameter(1, Z1, E_cm)
    E_G = gamow_energy(1, Z1, mu_MeV)

    results.append({
        'reaction': f"{target}(p,n){product}",
        'Q': Q,
        'E_lab': E_lab,
        'E_cm': E_cm,
        'Coulomb': V_c,
        'mu_MeV': mu_MeV,
        'mu_amu': mu_amu,
        'E_ratio': ratio,
        'eta': eta,
        'E_G': E_G
    })

In [ ]:
# Print results in a table
print("{:<15} {:<10} {:<10} {:<10} {:<10} {:<12} {:<10} {:<10} {:<10} {:<10}".format(
    "Reaction", "Q (MeV)", "E_lab", "E_cm", "V_c (MeV)",
    "μ (MeV/c²)", "μ (amu)", "E_cm/E_lab", "η", "E_G (MeV)"
))

for res in results:
    print("{:<15} {:>8.3f} {:>9.3f} {:>9.3f} {:>9.3f} {:>11.3f} {:>9.5f} {:>9.5f} {:>9.3f} {:>9.3f}".format(
        res['reaction'], res['Q'], res['E_lab'], res['E_cm'], res['Coulomb'],
        res['mu_MeV'], res['mu_amu'], res['E_ratio'], res['eta'], res['E_G']
    ))

Reaction        Q (MeV)    E_lab      E_cm       V_c (MeV)  μ (MeV/c²)   μ (amu)    E_cm/E_lab η          E_G (MeV) 
45Sc(p,n)45Ti     -3.359     3.434     3.359     5.530     917.710   0.98520   0.97809     1.791   425.414
48Ti(p,n)48V      -5.309     5.421     5.309     5.697     918.967   0.98655   0.97942     1.493   467.533
51V(p,n)51Cr      -5.254     5.358     5.254     5.862     920.080   0.98775   0.98061     1.571   511.622
52Cr(p,n)52Mn     -6.005     6.121     6.005     6.085     920.422   0.98811   0.98098     1.533   557.285
55Mn(p,n)55Fe     -5.250     5.346     5.250     6.246     921.379   0.98914   0.98200     1.709   605.321
56Fe(p,n)56Co     -5.860     5.965     5.860     6.465     921.675   0.98946   0.98231     1.683   654.925
59Co(p,n)59Ni     -2.366     2.407     2.366     6.622     922.505   0.99035   0.98320     2.751   706.909
62Ni(p,n)62Cu     -6.615     6.723     6.615     6.777     923.255   0.99116   0.98400     1.707   760.861
63Cu(p,n)63Zn     -4.660   